# Solutions · 17. Capstone: a soft sensor with a carbon balance, intervals and an alarm

Worked solutions to the three exercises of [Lecture 17](../../notebooks/part6_guardrails/17_capstone_industrial_soft_sensor_and_reconciliation.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Stage 1-4 End-to-End SNV + PLS1 + KKT Carbon-Conserved Soft Sensor")
rng = np.random.default_rng(2026)
N_tr, N_cal, N_te, d_wl = 60, 60, 80, 100
N_tot = N_tr + N_cal + N_te
wl = np.linspace(1200.0, 2200.0, d_wl)

# True carbon fractions [Y_MeOH, Y_CO, Y_CO2] summing identically to 1.0:
y_meoh = rng.uniform(0.15, 0.45, size=N_tot)
y_co = rng.uniform(0.05, 0.20, size=N_tot)
y_co2 = 1.0 - y_meoh - y_co
Y_true = np.column_stack([y_meoh, y_co, y_co2])

E_nir = np.vstack([
    np.exp(-0.5 * ((wl - 1480.0) / 55.0) ** 2) + 0.5 * np.exp(-0.5 * ((wl - 2050.0) / 45.0) ** 2),
    np.exp(-0.5 * ((wl - 1650.0) / 50.0) ** 2) + 0.4 * np.exp(-0.5 * ((wl - 1900.0) / 60.0) ** 2),
    np.exp(-0.5 * ((wl - 1780.0) / 65.0) ** 2) + 0.6 * np.exp(-0.5 * ((wl - 2120.0) / 50.0) ** 2),
])
scatter = rng.uniform(0.75, 1.30, size=(N_tot, 1))
offset = rng.uniform(-0.20, 0.20, size=(N_tot, 1))
X_raw = scatter * (Y_true @ E_nir) + offset + rng.normal(0.0, 0.012, size=(N_tot, d_wl))
X_snv = dim_reduction.snv_preprocess(X_raw)

Y_tr_obs = Y_true[:N_tr] + rng.normal(0.0, 0.018, size=(N_tr, 3))
pls_models = [dim_reduction.pls1_nipals(X_snv[:N_tr], Y_tr_obs[:, j], n_components=3) for j in range(3)]

def predict_raw_and_kkt(X_batch: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    Y_raw_pred = np.column_stack([X_batch @ m["coef"] + m["intercept"] for m in pls_models])
    C_c, d_c = np.array([[1.0, 1.0, 1.0]]), np.array([1.0])
    Y_kkt_pred = np.vstack([
        linear_physics.equality_constrained_lstsq(np.eye(3), row, C_c, d_c, lam=0.0)["w"]
        for row in Y_raw_pred
    ])
    return Y_raw_pred, Y_kkt_pred

Y_te_raw, Y_te_kkt = predict_raw_and_kkt(X_snv[N_tr + N_cal :])
Y_te_true = Y_true[N_tr + N_cal :]

max_carbon_err_raw = float(np.max(np.abs(np.sum(Y_te_raw, axis=1) - 1.0)))
max_carbon_err_kkt = float(np.max(np.abs(np.sum(Y_te_kkt, axis=1) - 1.0)))
mse_raw = float(np.mean((Y_te_raw - Y_te_true) ** 2))
mse_kkt = float(np.mean((Y_te_kkt - Y_te_true) ** 2))
print(f"Max carbon balance error: Raw PLS1 = {max_carbon_err_raw:.4e} -> KKT Reconciled = {max_carbon_err_kkt:.2e}")
print(f"Test MSE across 3 species: Raw PLS1 = {mse_raw:.6f} -> KKT Reconciled = {mse_kkt:.6f} ({100*(1-mse_kkt/mse_raw):.1f}% variance reduction)")
assert max_carbon_err_kkt < 1e-14 and mse_kkt < mse_raw
heading("Demonstration 2: Inverse-Variance Weighted Crowe Soft-Sensor Reconciliation")
y_raw_3 = np.array([0.32, 0.12, 0.60])  # Raw sum = 1.04 (+0.04 carbon surplus)
sig_3 = np.array([0.01, 0.02, 0.04])    # Variances proportional to [1, 4, 16]
rec_3 = linear_physics.crowe_data_reconciliation(y_raw_3, sig_3, np.array([[1.0, 1.0, 1.0]]), np.array([1.0]))
shifts = np.abs(rec_3["y_reconciled"] - y_raw_3)
print("Raw predictions [MeOH, CO, CO2]      :", y_raw_3, "(sum =", np.sum(y_raw_3), ")")
print("Crowe weighted reconciled predictions:", np.round(rec_3["y_reconciled"], 6), "(sum =", np.sum(rec_3["y_reconciled"]), ")")
print(f"Adjustment ratio |dY_CO2| / |dY_MeOH| = {shifts[2] / shifts[0]:.2f} (Theoretical (0.04/0.01)^2 = 16.00)")
assert np.isclose(shifts[2] / shifts[0], 16.0, rtol=1e-12) and np.max(np.abs(rec_3["balance_residual_after"])) < 1e-14
heading("Demonstration 3: Multi-Species 90% Split Conformal Prediction Intervals")
_, Y_cal_kkt = predict_raw_and_kkt(X_snv[N_tr : N_tr + N_cal])
Y_cal_obs = Y_true[N_tr : N_tr + N_cal] + rng.normal(0.0, 0.018, size=(N_cal, 3))
Y_te_obs = Y_te_true + rng.normal(0.0, 0.018, size=(N_te, 3))

for j, sp_name in enumerate(["CH3OH (Methanol)", "CO (Carbon Monoxide)", "CO2 (Unreacted Feed)"]):
    cp_j = uncertainty.split_conformal_interval(Y_cal_obs[:, j], Y_cal_kkt[:, j], Y_te_kkt[:, j], alpha=0.10)
    cov_j = float(np.mean((Y_te_obs[:, j] >= cp_j["lower"]) & (Y_te_obs[:, j] <= cp_j["upper"])))
    print(f"{sp_name:22s} | 90% Conformal Half-Width = +/-{cp_j['q_hat']:.4f} | Empirical Test Coverage = {cov_j*100:5.1f}%")
    assert cov_j >= 0.84
heading("Demonstration 4: Optical Window Fouling Sweep & PCA Q_SPE Automatic Interlock")
pca_guard = dim_reduction.pca_svd(X_snv[:N_tr], n_components=3)
q_limit_99 = float(np.percentile(pca_guard["q_spe"], 99))
P_g = pca_guard["loadings"]

print(f"{'Fouling Amp':>12s} | {'PCA Q_SPE':>11s} | {'Q / Q_99':>10s} | {'Unchecked MeOH Err':>18s} | {'PAT Interlock Decision':>24s}")
print("-" * 86)
for f_amp in [0.0, 0.2, 0.6, 1.2, 2.5]:
    x_f = X_snv[N_tr + N_cal].copy()
    x_f[40:50] += f_amp
    xc_f = x_f - pca_guard["mean"]
    q_f = float(np.sum((xc_f - (xc_f @ P_g) @ P_g.T) ** 2))
    _, y_kkt_f = predict_raw_and_kkt(x_f[None, :])
    meoh_err = abs(float(y_kkt_f[0, 0] - Y_te_true[0, 0]))
    decision = "HOLD / CLEAN PROBE" if q_f > q_limit_99 else "TRUST PREDICTION"
    print(f"{f_amp:12.2f} | {q_f:11.3f} | {q_f/q_limit_99:9.1f}x | {meoh_err:18.4f} | {decision:>24s}")
assert q_f > 10.0 * q_limit_99
heading("Inside the Algorithm: By-Hand Simplex Mean-Shift vs. the Library")
Y_kkt_hand = Y_te_raw - (np.sum(Y_te_raw, axis=1, keepdims=True) - 1.0) / 3.0
diff_cap = float(np.max(np.abs(Y_kkt_hand - Y_te_kkt)))
print("by hand (pure NumPy) row 0:", np.round(Y_kkt_hand[0], 6))
print("library (engml_2_applied)  row 0:", np.round(Y_te_kkt[0], 6))
print(f"Max discrepancy across 80x3 test matrix: {diff_cap:.2e}")
assert diff_cap < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Prove and verify on the $80 \times 3$ test set (`Y_te_raw`, `Y_te_kkt`, `Y_te_true`) the **exact per-sample Pythagorean error contraction**: because $\mathbf{1}_3^\top \mathbf{y}_{\mathrm{true}} = 1$, for *every single test sample* $i \in \{0, \dots, 79\}$, $\|\hat{\mathbf{y}}_{\mathrm{raw},i} - \mathbf{y}_{\mathrm{true},i}\|_2^2 - \|\hat{\mathbf{y}}_{\mathrm{KKT},i} - \mathbf{y}_{\mathrm{true},i}\|_2^2 = \frac{1}{3}(\mathbf{1}_3^\top \hat{\mathbf{y}}_{\mathrm{raw},i} - 1)^2 \ge 0$.

In [2]:
heading("Exercise 1: Exact Sample-by-Sample Pythagorean KKT Error Contraction Proof")
sq_raw_i = np.sum((Y_te_raw - Y_te_true) ** 2, axis=1)
sq_kkt_i = np.sum((Y_te_kkt - Y_te_true) ** 2, axis=1)
improvement_i = sq_raw_i - sq_kkt_i
theoretical_gain_i = ((np.sum(Y_te_raw, axis=1) - 1.0) ** 2) / 3.0
max_pyth_diff = float(np.max(np.abs(improvement_i - theoretical_gain_i)))
print(f"Fraction of test samples where KKT error <= Raw error: {100*np.mean(improvement_i >= -1e-15):.1f}% (Every single sample)")
print(f"Max discrepancy from (sum(y_raw) - 1)^2 / 3          : {max_pyth_diff:.2e}")
assert np.all(improvement_i >= -1e-15) and max_pyth_diff < 1e-14


Exercise 1: Exact Sample-by-Sample Pythagorean KKT Error Contraction Proof
Fraction of test samples where KKT error <= Raw error: 100.0% (Every single sample)
Max discrepancy from (sum(y_raw) - 1)^2 / 3          : 5.64e-18


**Discussion.** Because $\hat{\mathbf{y}}_{\mathrm{KKT},i}$ is the orthogonal Euclidean projection of $\hat{\mathbf{y}}_{\mathrm{raw},i}$ onto the hyperplane $\mathcal{H} = \{\mathbf{y} : \mathbf{1}_3^\top\mathbf{y} = 1\}$ and $\mathbf{y}_{\mathrm{true},i} \in \mathcal{H}$, the error vector decomposes orthogonally by the Pythagorean theorem on *every single sample*.

---

## Exercise 2

**Problem.** Audit the **dual-mode fault diagnostic Table** on the capstone soft sensor: test (1) a normal test spectrum, (2) a fouled probe window spectrum (`x_snv[40:50] += 2.0`), and (3) a stoichiometric composition drift (`Y_extr = [0.85, 0.10, 0.05]`, outside the training yield range $[0.15, 0.45]$). Verify that $Q_{\mathrm{SPE}}$ catches the optical probe fault while Hotelling's $T^2$ catches the extreme composition excursion.

In [3]:
heading("Exercise 2: Dual-Mode PAT Fault Matrix (Optical Fouling vs. Composition Excursion)")
t2_limit_99 = float(np.percentile(pca_guard["hotelling_t2"], 99))
lam_g = pca_guard["eigenvalues"]

x_norm_q = X_snv[N_tr + N_cal]
x_foul_q = x_norm_q.copy()
x_foul_q[40:50] += 2.0
x_extr_q = pca_guard["mean"] + (6.0 * np.sqrt(lam_g[0])) * P_g[:, 0]

for label, x_vec in [("Normal Reactor State", x_norm_q), ("Optical Window Fouling", x_foul_q), ("Extreme Yield Excursion", x_extr_q)]:
    xc = x_vec - pca_guard["mean"]
    t_v = xc @ P_g
    t2_v = float(np.sum((t_v ** 2) / lam_g))
    q_v = float(np.sum((xc - t_v @ P_g.T) ** 2))
    print(f"{label:24s} | T^2 = {t2_v:6.2f} (lim={t2_limit_99:.2f}) | Q_SPE = {q_v:7.3f} (lim={q_limit_99:.3f})")
    if label == "Optical Window Fouling":
        assert q_v > 5.0 * q_limit_99
    elif label == "Extreme Yield Excursion":
        assert t2_v > 2.5 * t2_limit_99 and q_v < q_limit_99


Exercise 2: Dual-Mode PAT Fault Matrix (Optical Fouling vs. Composition Excursion)
Normal Reactor State     | T^2 =   2.73 (lim=12.84) | Q_SPE =   0.615 (lim=1.072)
Optical Window Fouling   | T^2 =  59.70 (lim=12.84) | Q_SPE =  14.606 (lim=1.072)
Extreme Yield Excursion  | T^2 =  36.00 (lim=12.84) | Q_SPE =   0.000 (lim=1.072)


**Discussion.** Optical window fouling introduces spectral shape distortions orthogonal to the 3 Beer–Lambert chromophores (high $Q_{\mathrm{SPE}}$), whereas an extreme chemical conversion shift moves *along* the 3-chromophore Beer–Lambert subspace (high Hotelling $T^2$, low $Q_{\mathrm{SPE}}$).

---

## Exercise 3

**Problem.** Implement the complete **`IndustrialSoftSensorPAT`** class with `.fit(X_tr_raw, Y_tr, X_cal_raw, Y_cal)` and `.predict_guarded(X_query_raw)` methods that integrates Stage 1 (`snv_preprocess`), Stage 2 (PCA $T^2$ and $Q_{\mathrm{SPE}}$ 99% alarms), Stage 3 (3-species `pls1_nipals`), Stage 4 (inverse-variance `crowe_data_reconciliation` using calibration residual stds $\hat{\boldsymbol{\sigma}}_{\mathrm{cal}}$), and Stage 5 (90% `split_conformal_interval`). Verify it end-to-end on `X_raw[N_tr + N_cal:]`.

In [4]:
heading("Exercise 3: Implement End-to-End IndustrialSoftSensorPAT Pipeline Class")
class IndustrialSoftSensorPAT:
    """5-Stage Physics-Constrained & Fault-Guarded NIR Soft Sensor for Multicomponent Reactors."""

    def __init__(self, n_components: int = 3, alpha: float = 0.10) -> None:
        self.n_components = n_components
        self.alpha = alpha

    def fit(self, X_tr_raw: np.ndarray, Y_tr: np.ndarray, X_cal_raw: np.ndarray, Y_cal: np.ndarray) -> "IndustrialSoftSensorPAT":
        X_tr_s = dim_reduction.snv_preprocess(X_tr_raw)
        X_cal_s = dim_reduction.snv_preprocess(X_cal_raw)
        self.pca_ = dim_reduction.pca_svd(X_tr_s, n_components=self.n_components)
        self.t2_lim_ = float(np.percentile(self.pca_["hotelling_t2"], 99))
        self.q_lim_ = float(np.percentile(self.pca_["q_spe"], 99))
        self.pls_ = [dim_reduction.pls1_nipals(X_tr_s, Y_tr[:, j], n_components=self.n_components) for j in range(3)]
        Y_cal_raw = np.column_stack([X_cal_s @ m["coef"] + m["intercept"] for m in self.pls_])
        self.sig_cal_ = np.std(Y_cal_raw - Y_cal, axis=0)
        self.Y_cal_rec_ = self._reconcile_batch(Y_cal_raw)
        self.Y_cal_ = Y_cal
        return self

    def _reconcile_batch(self, Y_raw_b: np.ndarray) -> np.ndarray:
        A_c, b_c = np.array([[1.0, 1.0, 1.0]]), np.array([1.0])
        return np.vstack([linear_physics.crowe_data_reconciliation(row, self.sig_cal_, A_c, b_c)["y_reconciled"] for row in Y_raw_b])

    def predict_guarded(self, X_query_raw: np.ndarray) -> dict:
        Xq_s = dim_reduction.snv_preprocess(X_query_raw)
        Xc = Xq_s - self.pca_["mean"]
        T_q = Xc @ self.pca_["loadings"]
        t2_q = np.sum((T_q ** 2) / self.pca_["eigenvalues"][None, :], axis=1)
        q_q = np.sum((Xc - T_q @ self.pca_["loadings"].T) ** 2, axis=1)
        Y_raw_q = np.column_stack([Xq_s @ m["coef"] + m["intercept"] for m in self.pls_])
        Y_rec_q = self._reconcile_batch(Y_raw_q)
        intervals = [
            uncertainty.split_conformal_interval(self.Y_cal_[:, j], self.Y_cal_rec_[:, j], Y_rec_q[:, j], alpha=self.alpha)
            for j in range(3)
        ]
        return {"Y_reconciled": Y_rec_q, "intervals": intervals, "t2": t2_q, "q_spe": q_q, "q_alarm": q_q > self.q_lim_}

pat = IndustrialSoftSensorPAT(n_components=3, alpha=0.10).fit(
    X_raw[:N_tr], Y_tr_obs, X_raw[N_tr : N_tr + N_cal], Y_cal_obs
)
out_te = pat.predict_guarded(X_raw[N_tr + N_cal :])
carbon_closure_err = float(np.max(np.abs(np.sum(out_te["Y_reconciled"], axis=1) - 1.0)))
meoh_cov = float(np.mean((Y_te_obs[:, 0] >= out_te["intervals"][0]["lower"]) & (Y_te_obs[:, 0] <= out_te["intervals"][0]["upper"])))
print("IndustrialSoftSensorPAT deployed on 80 test spectra:")
print(f"  Max carbon balance residual = {carbon_closure_err:.2e} | Methanol 90% Conformal Coverage = {meoh_cov*100:.1f}%")
assert carbon_closure_err < 1e-14 and meoh_cov >= 0.85


Exercise 3: Implement End-to-End IndustrialSoftSensorPAT Pipeline Class
IndustrialSoftSensorPAT deployed on 80 test spectra:
  Max carbon balance residual = 2.22e-16 | Methanol 90% Conformal Coverage = 93.8%


**Discussion.** Combining spectral SNV preprocessing, supervised NIPALS PLS1, covariance-weighted Crowe/KKT stoichiometric reconciliation, MSPC $T^2/Q_{\mathrm{SPE}}$ fault alarms, and split conformal prediction gives a soft sensor whose predictions satisfy the carbon balance, carry an interval, and are withheld when the spectrum is unlike those of the calibration.